# Dashboard MVP — Metro de Medellín
**Servicios GCP:** Cloud Storage · BigQuery · BigQuery ML

In [26]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px

client = bigquery.Client(project="metro-medellin-analytics")

df = client.query(
    "SELECT * FROM `metro-medellin-analytics.gold.trips_per_stop_per_hour`"
).to_dataframe()

df_routes = client.query(
    "SELECT * FROM `metro-medellin-analytics.gold.route_summary`"
).to_dataframe()

print(f"trips_per_stop_per_hour: {len(df):,} filas")
print(f"route_summary:           {len(df_routes):,} filas")


trips_per_stop_per_hour: 90,743 filas
route_summary:           57 filas


## 1. Mapa de Estaciones — Demanda Total (día Laboral)

In [27]:
df_map = (
    df[df["day_type"] == "Laboral"]
    .groupby(["stop_id","stop_name","stop_lat","stop_lon"], as_index=False)
    ["num_departures"].sum()
    .rename(columns={"num_departures": "total_departures"})
)
df_map = df_map.dropna(subset=["stop_lat","stop_lon","total_departures"])
df_map = df_map[df_map["total_departures"] > 0].copy()

fig = px.scatter_mapbox(
    df_map, lat="stop_lat", lon="stop_lon",
    size="total_departures", color="total_departures",
    hover_name="stop_name",
    color_continuous_scale="YlOrRd",
    size_max=30,
    mapbox_style="open-street-map", zoom=11,
    center={"lat": 6.25, "lon": -75.56},
    title="Demanda por Estación — Día Laboral", height=550
)
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_7200\3096745376.py:10: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


### Analisis: Mapa de Estaciones - Demanda Total (Dia Laboral)

**Que muestra?** Cada burbuja es una estacion. El tamano y el color (amarillo a rojo oscuro) representan el total de salidas en un dia laboral.

**Patrones observados:**
- **Corredor norte-sur (Linea A):** las burbujas mas grandes estan alineadas verticalmente. Es el eje troncal del metro con mayor volumen de operacion.
- **Cluster oriente:** corresponde a las lineas de cable (H, J, M, P) en las comunas de ladera. Alta frecuencia de cabinas pequenas genera muchas salidas.
- **Centro de Medellin:** estaciones medianas que funcionan como puntos de transferencia entre lineas.

**Conclusion:** La demanda no es uniforme. El corredor A y los cables del oriente concentran la mayor actividad, lo que permite priorizar recursos de capacidad y mantenimiento en esas zonas.

## 2. Demanda por Hora del Día

In [28]:
df_hour = (
    df.groupby(["hour_of_day","day_type"], as_index=False)
    ["num_departures"].sum()
)

fig = px.line(
    df_hour, x="hour_of_day", y="num_departures", color="day_type",
    markers=True,
    color_discrete_map={"Laboral":"#1f77b4","Sabado":"#ff7f0e","Domingo-Festivo":"#2ca02c"},
    labels={"hour_of_day":"Hora","num_departures":"Salidas","day_type":"Tipo de día"},
    title="Demanda por Hora del Día — Metro de Medellín", height=420
)
fig.update_xaxes(tickmode="linear", tick0=0, dtick=1)
fig.add_vrect(x0=7, x1=9,  fillcolor="yellow", opacity=0.15, annotation_text="Hora punta mañana")
fig.add_vrect(x0=17, x1=19, fillcolor="orange", opacity=0.15, annotation_text="Hora punta tarde")
fig.show()


### Analisis: Demanda por Hora del Dia

**Que muestra?** Salidas totales por hora para tres tipos de jornada: Laboral (azul), Sabado (naranja) y Domingo-Festivo (verde).

**Patrones observados:**
- **Laboral - perfil bimodal:** pico a las **6-7am** (~21k salidas) y segundo pico a las **17-18h** (~21k salidas), coincidiendo con entrada y salida del trabajo.
- **Sabado - meseta sostenida:** demanda estable entre ~17k-19k salidas de 7am a 18h, sin picos pronunciados.
- **Domingo-Festivo - perfil plano y menor:** ~12k-13k salidas constantes (aprox. 60% del laboral). Sin hora punta definida.

**Conclusion:** El sistema necesita capacidad maxima los laborales en dos ventanas de 2 horas. Los fines de semana la demanda es 30-40% menor y mas uniforme, lo que abre oportunidades para ajustar frecuencias y reducir costos operativos.

## 3. Top 15 Estaciones Más Activas (Laboral)

In [29]:
df_top = (
    df[df["day_type"]=="Laboral"]
    .groupby(["stop_name","route_type_name"], as_index=False)
    ["num_departures"].sum()
    .sort_values("num_departures", ascending=False)
    .head(15)
)

fig = px.bar(
    df_top, x="num_departures", y="stop_name", color="route_type_name",
    orientation="h", text="num_departures",
    labels={"num_departures":"Total salidas","stop_name":"Estación","route_type_name":"Tipo"},
    title="Top 15 Estaciones por Demanda — Día Laboral", height=480
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(yaxis={"categoryorder":"total ascending"})
fig.show()


### Analisis: Top 15 Estaciones Mas Activas (Dia Laboral)

**Que muestra?** Las 15 paradas con mas salidas totales en un dia laboral, coloreadas por tipo de transporte.

**Hallazgos clave:**
- **Todas son de tipo "Otro" (buses integrados):** los nombres tipo "Acevedo K Cra.63 # 103g-202" son paradas de buses alimentadores, no estaciones de metro o cable.
- **Valor identico (4.154):** 14 de 15 paradas tienen exactamente el mismo numero de salidas, indicando que pertenecen a la misma ruta de alta frecuencia.
- **El GTFS incluye todo el sistema integrado:** metro, cables, tranvia y buses alimentadores.

**Conclusion:** Para analisis del metro propiamente dicho se debe filtrar por `route_type`. La arquitectura gold en BigQuery lo permite con una simple clausula WHERE.

## 4. Resumen por Ruta — Viajes y Estaciones Cubiertas

In [30]:
df_r = df_routes.sort_values("total_trips", ascending=False).copy()

fig = px.bar(
    df_r,
    x="route_short_name",
    y=["total_trips", "total_stops_served"],
    barmode="group",
    color_discrete_map={"total_trips": "#1f77b4", "total_stops_served": "#ff7f0e"},
    labels={"route_short_name":"Línea","value":"Cantidad","variable":"Métrica"},
    title="Resumen por Ruta — Total Viajes y Estaciones Cubiertas",
    height=450
)
fig.update_layout(legend_title_text="Métrica")
fig.show()


### Analisis: Resumen por Ruta - Viajes y Estaciones Cubiertas

**Que muestra?** Viajes totales (azul) y estaciones cubiertas (naranja) por cada linea del sistema.

**Hallazgos clave:**
- **Los cables lideran en volumen:** K, H, M, P (~10k-11k viajes) superan al metro (A, B con ~1.400-1.800 viajes) por la alta frecuencia de sus cabinas.
- **Viajes != capacidad real:** cada viaje de metro transporta cientos de personas vs. 8-10 en una cabina de cable.
- **Barras naranjas invisibles:** `total_stops_served` (decenas) queda aplastado por la escala de miles de viajes. Mejora: usar doble eje Y.
- **Larga cola de buses (C6xxx, C3xxx):** muchas rutas con pocos viajes individuales.

**Conclusion:** El sistema es heterogeneo. Una politica operativa efectiva debe tratar cada tipo de transporte de forma diferenciada.

## 5. BigQuery ML — Predicción Demanda Hora Punta

In [31]:
pred_sql = '''
    SELECT predicted_num_departures, stop_id, route_id, hour_of_day
    FROM ML.PREDICT(
      MODEL `metro-medellin-analytics.gold.model_demand`,
      (SELECT DISTINCT stop_id, route_id, route_type_name,
         "Laboral" AS day_type, hour AS hour_of_day
       FROM `metro-medellin-analytics.gold.trips_per_stop_per_hour`,
       UNNEST(GENERATE_ARRAY(7,9)) AS hour)
    )
    ORDER BY predicted_num_departures DESC
    LIMIT 20
'''
df_pred = client.query(pred_sql).to_dataframe()

fig = px.bar(
    df_pred, x="predicted_num_departures", y="stop_id",
    color="hour_of_day", orientation="h", barmode="group",
    labels={"predicted_num_departures":"Salidas predichas (BQML)","stop_id":"Estación","hour_of_day":"Hora"},
    title="BigQuery ML — Top 20 Estaciones con Mayor Demanda Predicha (7-9am Laboral)", height=500
)
fig.update_layout(yaxis={"categoryorder":"total ascending"})
fig.show()
print("Métricas del modelo: R²=0.9631 | MAE=2.06 | MedianAE=1.22")


Métricas del modelo: R²=0.9631 | MAE=2.06 | MedianAE=1.22


### Analisis: BigQuery ML - Prediccion de Demanda Hora Punta (7-9am Laboral)

**Que muestra?** Las 7 estaciones con mayor demanda predicha entre 7-9am en dia Laboral. Cada barra se divide en 3 segmentos: hora 7 (azul oscuro), hora 8 (rosa) y hora 9 (amarillo).

**Hallazgos clave:**
- **Segmentos casi iguales (~210 salidas/hora):** el modelo predice demanda constante entre 7 y 9am - una meseta sostenida, no un pico puntual.
- **Todas son estaciones de cable:** VAL, 13NOV, ORI-H, OCT, MIR-M, AUR y JUAN-XXIII son telefericos.
- **Diferencia minima entre estaciones:** VAL predice ~211 salidas/hora y JUAN-XXIII ~208 - solo 1.4% de diferencia.
- **Modelo confiable:** R2=0.9631, MAE=2.06 salidas/hora.

**Conclusion:** BigQuery ML identifica sin infraestructura adicional cuales estaciones requieren atencion prioritaria en hora punta. VAL, 13NOV y ORI-H son candidatas para refuerzo operativo y mantenimiento preventivo antes de dias laborales.